###Libraries

In [ ]:
!pip install transformers datasets seqeval Pillow torch torchvision -q
print("✅ All libraries installed!")

In [ ]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu -q

In [ ]:
# Standard libraries
import os
import json
import random
import numpy as np
import pandas as pd

# Image handling
from PIL import Image

# PyTorch
import torch
from torch.utils.data import Dataset, DataLoader

# HuggingFace
from transformers import (
    LayoutLMv3Processor,
    LayoutLMv3ForTokenClassification,
    LayoutLMv3ForSequenceClassification,
    TrainingArguments,
    Trainer
)
from datasets import load_from_disk


# Evaluation
import seqeval
from seqeval.metrics import (
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

# Visualization (for checking data)
import matplotlib.pyplot as plt
import matplotlib.patches as patches

print("✅ All imports done!")

In [ ]:
import os

# Project paths
PROJECT_DIR     = '/content/drive/MyDrive/DocFusion'
SROIE_TRAIN     = f'{PROJECT_DIR}/data/SROIE2019/train'
SROIE_TEST      = f'{PROJECT_DIR}/data/SROIE2019/test'
CORD_DIR        = f'{PROJECT_DIR}/data/cord'
FINDITAGAIN_DIR = f'{PROJECT_DIR}/data/findit2'
MODELS_DIR      = f'{PROJECT_DIR}/models'

os.makedirs(MODELS_DIR, exist_ok=True)
print("✅ Paths set!")

In [ ]:
import torch

if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"✅ Using GPU: {torch.cuda.get_device_name(0)}")
else:
    device = torch.device("cpu")
    print("⚠️ Using CPU")

###Load & Parse SROIE Data

In [ ]:
SROIE_TRAIN_ENTITIES = f'{SROIE_TRAIN}/entities'
receipt_ids = [f.replace('.txt', '') for f in os.listdir(SROIE_TRAIN_ENTITIES) if f.endswith('.txt')]

print(receipt_ids)
print("totl number of receipts: ", len(receipt_ids))


In [ ]:
def parse_box_file(box_path):
    words = []
    boxes = []

    with open(box_path, 'r', encoding='utf-8', errors='ignore') as f:
        content = f.read()
        lines = content.splitlines()
        for line in lines:
            line = line.strip()
            if not line:
                continue
            part = line.split(',', 8)

            if len(part) < 9:
                continue

            x1, y1, x2, y2, x3, y3, x4, y4 = [int(p) for p in part[:8]]
            text = part[8].strip()
            x_min = min(x1, x4)
            y_min = min(y1, y2)
            x_max = max(x2, x3)
            y_max = max(y3, y4)

            boxes.append([x_min, y_min, x_max, y_max])
            words.append(text)

    return words, boxes

In [ ]:

def parse_entity_file(entity_path):
    try:
        with open(entity_path, 'r') as f:
            data = json.load(f)

        return {
            "vendor": data.get("company", None),
            "date":   data.get("date", None),
            "total":  data.get("total", None)
        }
    except:
        return {
            "vendor": None,
            "date":   None,
            "total":  None
        }



In [ ]:
def normalize_boxes(boxes, width, height):
    normalized = []

    for box in boxes:
        x_min, y_min, x_max, y_max = box

        normalized_x_min = min(int(x_min / width  * 1000), 1000)
        normalized_y_min = min(int(y_min / height * 1000), 1000)
        normalized_x_max = min(int(x_max / width  * 1000), 1000)
        normalized_y_max = min(int(y_max / height * 1000), 1000)

        normalized.append([normalized_x_min, normalized_y_min, normalized_x_max, normalized_y_max])

    return normalized

In [ ]:
sroie_data = []

for receipt_id in receipt_ids:
    image_path  = f'{SROIE_TRAIN}/img/{receipt_id}.jpg'
    if not os.path.exists(image_path):
        image_path = f'{SROIE_TRAIN}/img/{receipt_id}.png'

    box_path    = f'{SROIE_TRAIN}/box/{receipt_id}.txt'
    entity_path = f'{SROIE_TRAIN}/entities/{receipt_id}.txt'

    if not os.path.exists(image_path) or \
       not os.path.exists(box_path) or \
       not os.path.exists(entity_path):
        continue

    with Image.open(image_path) as img:
        width, height = img.size

    words, boxes = parse_box_file(box_path)
    entity       = parse_entity_file(entity_path)

    sroie_data.append({
        'id':         receipt_id,
        'image_path': image_path,
        'words':      words,
        'boxes':      normalize_boxes(boxes, width, height),
        'vendor':     entity['vendor'],
        'date':       entity['date'],
        'total':      entity['total']
    })

print(f"Total receipts loaded: {len(sroie_data)}")
print(f"\nSample receipt:")
print(f"  ID:     {sroie_data[0]['id']}")
print(f"  Words:  {len(sroie_data[0]['words'])}")
print(f"  Vendor: {sroie_data[0]['vendor']}")
print(f"  Date:   {sroie_data[0]['date']}")
print(f"  Total:  {sroie_data[0]['total']}")

In [ ]:
test_receipt_ids = [
    f.replace('.txt', '')
    for f in os.listdir(f'{SROIE_TEST}/entities')
    if f.endswith('.txt')
]
print(f"Test receipt IDs found: {len(test_receipt_ids)}")

In [ ]:
sroie_test_data = []

for receipt_id in test_receipt_ids:
    image_path  = f'{SROIE_TEST}/img/{receipt_id}.jpg'
    if not os.path.exists(image_path):
        image_path = f'{SROIE_TEST}/img/{receipt_id}.png'

    box_path    = f'{SROIE_TEST}/box/{receipt_id}.txt'      # ← SROIE_TEST
    entity_path = f'{SROIE_TEST}/entities/{receipt_id}.txt' # ← SROIE_TEST

    if not os.path.exists(image_path) or \
       not os.path.exists(box_path) or \
       not os.path.exists(entity_path):
        continue

    with Image.open(image_path) as img:
        width, height = img.size

    words, boxes = parse_box_file(box_path)
    entity       = parse_entity_file(entity_path)

    sroie_test_data.append({
        'id':         receipt_id,
        'image_path': image_path,
        'words':      words,
        'boxes':      normalize_boxes(boxes, width, height),
        'vendor':     entity['vendor'],
        'date':       entity['date'],
        'total':      entity['total']
    })

print(f"SROIE test receipts loaded: {len(sroie_test_data)}")

In [ ]:
print(f"Total receipts loaded: {len(sroie_test_data)}")
print(f"\nSample receipt:")
print(f"  ID:     {sroie_test_data[0]['id']}")
print(f"  Words:  {len(sroie_test_data[0]['words'])}")
print(f"  Vendor: {sroie_test_data[0]['vendor']}")
print(f"  Date:   {sroie_test_data[0]['date']}")
print(f"  Total:  {sroie_test_data[0]['total']}")

###Load & Parse CORD Data

In [ ]:
from datasets import load_from_disk

cord = load_from_disk(CORD_DIR)
print(f"CORD loaded")
print(f"Splits: {cord}")

In [ ]:
from pprint import pprint
sample = cord['train'][0]

print("Image:")
print(sample['image'])

print("\nGround Truth (raw):")
pprint(sample['ground_truth'])

In [ ]:
def parse_cord_ground_truth(ground_truth_str):
    try:
        data     = json.loads(ground_truth_str)
        gt_parse = data.get("gt_parse", {})


        total_dict = gt_parse.get("total", {})
        if isinstance(total_dict, dict):
            total = total_dict.get("total_price", None)
        else:
            total = None

        return {
            "vendor": None,
            "date":   None,
            "total":  str(total) if total else None
        }
    except:
        return {"vendor": None, "date": None, "total": None}

In [ ]:
def parse_cord_words_boxes(ground_truth_str, img_width, img_height):
    try:
        data       = json.loads(ground_truth_str)
        valid_line = data.get("valid_line", [])

        words = []
        boxes = []

        for line in valid_line:
            for word in line.get("words", []):
                text = word.get("text", "").strip()
                quad = word.get("quad", {})

                if not text:
                    continue


                x1 = quad.get("x1", 0)
                y1 = quad.get("y1", 0)
                x2 = quad.get("x2", 0)
                y2 = quad.get("y2", 0)
                x3 = quad.get("x3", 0)
                y3 = quad.get("y3", 0)
                x4 = quad.get("x4", 0)
                y4 = quad.get("y4", 0)


                x_min = min(x1, x4)
                y_min = min(y1, y2)
                x_max = max(x2, x3)
                y_max = max(y3, y4)


                x_min = min(int(x_min / img_width  * 1000), 1000)
                y_min = min(int(y_min / img_height * 1000), 1000)
                x_max = min(int(x_max / img_width  * 1000), 1000)
                y_max = min(int(y_max / img_height * 1000), 1000)

                words.append(text)
                boxes.append([x_min, y_min, x_max, y_max])

        return words, boxes
    except:
        return [], []

In [ ]:
cord_data = []

for idx, sample in enumerate(cord['train']):
    try:
        image         = sample['image']
        width, height = image.size

        words, boxes = parse_cord_words_boxes(
            sample['ground_truth'], width, height
        )

        if not words:
            continue

        entity = parse_cord_ground_truth(sample['ground_truth'])

        cord_data.append({
            "id":         f"cord_train_{idx}",
            "image_path": None,
            "image":      image,
            "words":      words,
            "boxes":      boxes,
            "vendor":     entity["vendor"],
            "date":       entity["date"],
            "total":      entity["total"]
        })

    except Exception as e:
        print(f"Skipping sample {idx}: {e}")
        continue

print(f"CORD train samples loaded: {len(cord_data)}")

In [ ]:
cord_val_data  = []
cord_test_data = []

for idx, sample in enumerate(cord['validation']):
    try:
        image         = sample['image']
        width, height = image.size
        words, boxes  = parse_cord_words_boxes(sample['ground_truth'], width, height)
        if not words:
            continue
        entity = parse_cord_ground_truth(sample['ground_truth'])
        cord_val_data.append({
            "id": f"cord_val_{idx}", "image_path": None,
            "image": image, "words": words, "boxes": boxes,
            "vendor": entity["vendor"], "date": entity["date"],
            "total": entity["total"]
        })
    except:
        continue

for idx, sample in enumerate(cord['test']):
    try:
        image         = sample['image']
        width, height = image.size
        words, boxes  = parse_cord_words_boxes(sample['ground_truth'], width, height)
        if not words:
            continue
        entity = parse_cord_ground_truth(sample['ground_truth'])
        cord_test_data.append({
            "id": f"cord_test_{idx}", "image_path": None,
            "image": image, "words": words, "boxes": boxes,
            "vendor": entity["vendor"], "date": entity["date"],
            "total": entity["total"]
        })
    except:
        continue

print(f"CORD val  samples: {len(cord_val_data)}")
print(f"CORD test samples: {len(cord_test_data)}")

In [ ]:
print("=" * 45)
print("📊 Data Loading Summary")
print("=" * 45)
print(f"SROIE Train:  {len(sroie_data)}")
print(f"SROIE Test:   {len(sroie_test_data)}")
print(f"CORD Train:   {len(cord_data)}")
print(f"CORD Val:     {len(cord_val_data)}")
print(f"CORD Test:    {len(cord_test_data)}")
print("=" * 45)

###Checking for both datasets

In [ ]:
print("=" * 50)
print("📊 SROIE Train — First 5 Samples")
print("=" * 50)

for i, sample in enumerate(sroie_data[:5]):
    print(f"\n🔍 Sample {i+1}:")
    print(f"  ID:        {sample['id']}")
    print(f"  Image:     {sample['image_path']}")
    print(f"  Words:     {sample['words'][:5]}...")
    print(f"  Boxes:     {sample['boxes'][:3]}...")
    print(f"  Vendor:    {sample['vendor']}")
    print(f"  Date:      {sample['date']}")
    print(f"  Total:     {sample['total']}")

In [ ]:
print("=" * 50)
print("📊 CORD Train — First 5 Samples")
print("=" * 50)

for i, sample in enumerate(cord_data[:5]):
    print(f"\n🔍 Sample {i+1}:")
    print(f"  ID:        {sample['id']}")
    print(f"  Image:     {type(sample['image'])}")
    print(f"  Words:     {sample['words'][:5]}...")
    print(f"  Boxes:     {sample['boxes'][:3]}...")
    print(f"  Vendor:    {sample['vendor']}")
    print(f"  Date:      {sample['date']}")
    print(f"  Total:     {sample['total']}")


In [ ]:
# Check first 3 receipt IDs and their box paths
for receipt_id in list(receipt_ids[:3]):
    box_path = f'{SROIE_TRAIN}/box/{receipt_id}.txt'
    print(f"ID: {receipt_id}")
    print(f"Box path exists: {os.path.exists(box_path)}")
    print(f"Box path: {box_path}")
    print()

In [ ]:
# Check what files are actually in the box folder
box_files = os.listdir(f'{SROIE_TRAIN}/box')
print(f"First 5 box files: {box_files[:5]}")
print(f"Total box files: {len(box_files)}")

In [ ]:
# Test parse_box_file on 3 different receipts
for receipt_id in list(receipt_ids[:3]):
    box_path = f'{SROIE_TRAIN}/box/{receipt_id}.txt'
    words, boxes = parse_box_file(box_path)
    print(f"ID: {receipt_id}")
    print(f"First word: {words[0]}")
    print()

###Combining the datasets

In [ ]:
import random
all_train_data = sroie_data + cord_data
all_val_data = sroie_test_data + cord_val_data
random.shuffle(all_train_data)

print("training data size: ",len(all_train_data))
print("validataion data size: ",len(all_val_data))

###Tagging

In [ ]:
labels = ["O", "B-VENDOR", "I-VENDOR", "B-DATE", "I-DATE", "B-TOTAL", "I-TOTAL"]

label2id = {x: index for index, x in enumerate(labels, start=0)}


id2label = {index: x for index, x in enumerate(labels, start=0)}


In [ ]:
label2id

In [ ]:
id2label

In [ ]:
def word_in_entity(word, entity_value):
    if entity_value is None:
        return False

    word_lower   = word.lower().strip()
    entity_lower = entity_value.lower().strip()


    if len(word_lower) < 3:
        return False


    return word_lower in entity_lower or entity_lower in word_lower


In [ ]:
def assign_bio_labels(words, vendor, date, total):
    label_ids      = []
    vendor_started = False
    date_started   = False
    total_started  = False

    for word in words:

        if word_in_entity(word, vendor):
            if not vendor_started:
                label_ids.append(label2id['B-VENDOR'])
                vendor_started = True
            else:
                label_ids.append(label2id['I-VENDOR'])

        elif word_in_entity(word, date):
            if not date_started:
                label_ids.append(label2id['B-DATE'])
                date_started = True
            else:
                label_ids.append(label2id['I-DATE'])

        elif word_in_entity(word, total):
            if not total_started:
                label_ids.append(label2id['B-TOTAL'])
                total_started = True
            else:
                label_ids.append(label2id['I-TOTAL'])

        else:
            label_ids.append(label2id['O'])

    return label_ids

In [ ]:
sample = sroie_data[0]
label_ids = assign_bio_labels(
    sample['words'],
    sample['vendor'],
    sample['date'],
    sample['total']
)

In [ ]:
for word, label_id in zip(sample['words'], label_ids):
    print(f"{word:<30} → {id2label[label_id]}")

In [ ]:
for sample in all_train_data:
    sample['labels'] = assign_bio_labels(
        sample['words'],
        sample['vendor'],
        sample['date'],
        sample['total']
    )

for sample in all_val_data:
    sample['labels'] = assign_bio_labels(
        sample['words'],
        sample['vendor'],
        sample['date'],
        sample['total']
    )

print(f"BIO tags applied to {len(all_train_data)} training samples")
print(f"BIO tags applied to {len(all_val_data)} validation samples")

In [ ]:
# Check one SROIE sample
print("=" * 50)
print("📊 SROIE Sample Check")
print("=" * 50)
sample = next(s for s in all_train_data if s['vendor'] is not None)
print(f"ID:     {sample['id']}")
print(f"Vendor: {sample['vendor']}")
print(f"Date:   {sample['date']}")
print(f"Total:  {sample['total']}")
print(f"Words:  {len(sample['words'])}")
print(f"Labels: {len(sample['labels'])}")
print(f"Match:  {len(sample['words']) == len(sample['labels'])}")
print(f"\nLabel distribution:")
from collections import Counter
counts = Counter([id2label[l] for l in sample['labels']])
for label, count in counts.items():
    print(f"  {label}: {count}")

# Check one CORD sample
print("\n" + "=" * 50)
print("📊 CORD Sample Check")
print("=" * 50)
cord_sample = next(s for s in all_train_data if s['vendor'] is None)
print(f"ID:     {cord_sample['id']}")
print(f"Vendor: {cord_sample['vendor']}")
print(f"Date:   {cord_sample['date']}")
print(f"Total:  {cord_sample['total']}")
print(f"Words:  {len(cord_sample['words'])}")
print(f"Labels: {len(cord_sample['labels'])}")
print(f"Match:  {len(cord_sample['words']) == len(cord_sample['labels'])}")
print(f"\nLabel distribution:")
counts = Counter([id2label[l] for l in cord_sample['labels']])
for label, count in counts.items():
    print(f"  {label}: {count}")


In [ ]:
import pickle

with open(f'{PROJECT_DIR}/data/all_train_data.pkl', 'wb') as f:
    pickle.dump(all_train_data, f)

with open(f'{PROJECT_DIR}/data/all_val_data.pkl', 'wb') as f:
    pickle.dump(all_val_data, f)

print("✅ Data saved to Drive!")

###Initializing the model (LayoutLMv3) --> turns out to be bad idea

In [ ]:
import pickle
with open(f'{PROJECT_DIR}/data/all_train_data.pkl', 'rb') as f:
    all_train_data = pickle.load(f)

with open(f'{PROJECT_DIR}/data/all_val_data.pkl', 'rb') as f:
    all_val_data = pickle.load(f)

print(f"✅ Train: {len(all_train_data)}")
print(f"✅ Val:   {len(all_val_data)}")

In [ ]:

!apt-get install tesseract-ocr -q
print("Tesseract installed!")

In [ ]:
from transformers import LayoutLMv3Processor

processor = LayoutLMv3Processor.from_pretrained(
    "microsoft/layoutlmv3-base",
    apply_ocr=False
)
print("Processor loaded!")

In [ ]:
from transformers import LayoutLMv3ForTokenClassification

model = LayoutLMv3ForTokenClassification.from_pretrained(
    "microsoft/layoutlmv3-base",
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id
)
print("✅ Model loaded!")
print(f"   Parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")
print(f"   Labels: {list(label2id.keys())}")

In [ ]:
print("✅ Model Config:")
print(f"   Model type:  {model.config.model_type}")
print(f"   Num labels:  {model.config.num_labels}")
print(f"   Hidden size: {model.config.hidden_size}")
print(f"   id2label:    {model.config.id2label}")


In [ ]:
model

In [ ]:
from transformers import LayoutLMForTokenClassification, LayoutLMTokenizerFast

tokenizer = LayoutLMTokenizerFast.from_pretrained(
    "microsoft/layoutlm-base-uncased"
)

model = LayoutLMForTokenClassification.from_pretrained(
    "microsoft/layoutlm-base-uncased",
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id
)
print("LayoutLM v1 loaded!")

In [ ]:
def process_sample(sample):
    encoding = tokenizer(
        sample['words'],
        boxes=sample['boxes'],
        word_labels=sample['labels'],
        padding='max_length',
        truncation=True,
        max_length=512,
        return_tensors='pt',
    )
    return {k: v.squeeze(0) for k, v in encoding.items()}

In [ ]:
sample   = next(s for s in all_train_data if s.get('image_path') is not None)
encoding = process_sample(sample)

print("Processing works!")
print(f"Keys:           {list(encoding.keys())}")
print(f"input_ids:      {encoding['input_ids'].shape}")
print(f"attention_mask: {encoding['attention_mask'].shape}")
print(f"bbox:           {encoding['bbox'].shape}")
print(f"pixel_values:   {encoding['pixel_values'].shape}")
print(f"labels:         {encoding['labels'].shape}")

In [ ]:
sample   = next(s for s in all_train_data if s.get('image_path') is None)
encoding = process_sample(sample)

print("CORD processing works!")
print(f"input_ids:      {encoding['input_ids'].shape}")
print(f"pixel_values:   {encoding['pixel_values'].shape}")
print(f"labels:         {encoding['labels'].shape}")

In [ ]:
sample   = next(s for s in all_train_data if s.get('vendor') is not None)
encoding = process_sample(sample)

tokens = processor.tokenizer.convert_ids_to_tokens(encoding['input_ids'])
print("📊 Token → Label mapping (first 20):")
for token, label_id in zip(tokens[:20], encoding['labels'][:20].tolist()):
    label = id2label.get(label_id, 'IGN') if label_id != -100 else '-100'
    print(f"  {token:<20} → {label}")

###Pytorch dataset

In [ ]:
class DocFusionDataset(Dataset):
    def __init__(self, data):
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        sample   = self.data[idx]
        encoding = process_sample(sample)
        return encoding

In [ ]:
train_dataset = DocFusionDataset(all_train_data)
val_dataset   = DocFusionDataset(all_val_data)

print(f"Train dataset: {len(train_dataset)} samples")
print(f"Val dataset:   {len(val_dataset)} samples")

In [ ]:
sample = train_dataset[0]

print("Dataset item test:")
print(f"  input_ids:      {sample['input_ids'].shape}")
print(f"  attention_mask: {sample['attention_mask'].shape}")
print(f"  bbox:           {sample['bbox'].shape}")
print(f"  pixel_values:   {sample['pixel_values'].shape}")
print(f"  labels:         {sample['labels'].shape}")

In [ ]:
!pip install seqeval -q


In [ ]:
import numpy as np
from seqeval.metrics import precision_score, recall_score, f1_score, classification_report

def compute_metrics(pred):
    predictions, labels = pred
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [id2label[p] for p, l in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [id2label[l] for p, l in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    return {
        "precision": precision_score(true_labels, true_predictions),
        "recall":    recall_score(true_labels, true_predictions),
        "f1":        f1_score(true_labels, true_predictions),
    }



In [ ]:
training_args = TrainingArguments(
    output_dir                  = f'{MODELS_DIR}/docfusion-v1',
    num_train_epochs            = 5,
    per_device_train_batch_size = 2,
    per_device_eval_batch_size  = 2,
    learning_rate               = 5e-5,
    weight_decay                = 0.01,
    eval_strategy               = "epoch",
    save_strategy               = "epoch",
    load_best_model_at_end      = True,
    metric_for_best_model       = "f1",
    save_total_limit            = 2,
    logging_steps               = 50,
    fp16                        = False,
    bf16                        = True,
    push_to_hub                 = False,
    report_to                   = "none"
)



In [ ]:
from transformers import Trainer

trainer = Trainer(
    model           = model,
    args            = training_args,
    train_dataset   = train_dataset,
    eval_dataset    = val_dataset,
    compute_metrics = compute_metrics
)

print("Trainer created!")
print(f"   Train samples: {len(train_dataset)}")
print(f"   Val samples:   {len(val_dataset)}")
print(f"   Epochs:        {training_args.num_train_epochs}")
print(f"   Batch size:    {training_args.per_device_train_batch_size}")
print(f"   Total steps:   {len(train_dataset) // training_args.per_device_train_batch_size * training_args.num_train_epochs}")

In [ ]:

model.train()
batch  = next(iter(DataLoader(train_dataset, batch_size=2)))
inputs = {k: v.to(model.device) for k, v in batch.items()}

with torch.no_grad():
    outputs = model(**inputs)

print(f"✅ Forward pass works!")
print(f"   Loss:   {outputs.loss.item():.4f}")
print(f"   Logits: {outputs.logits.shape}")


In [ ]:
batch = next(iter(DataLoader(train_dataset, batch_size=2)))

print("📊 Label values:")
print(f"  Min: {batch['labels'].min().item()}")
print(f"  Max: {batch['labels'].max().item()}")
print(f"  Unique: {batch['labels'].unique().tolist()}")
print(f"  Any NaN: {torch.isnan(batch['labels'].float()).any()}")

In [ ]:
print("📊 Bbox values:")
print(f"  Min: {batch['bbox'].min().item()}")
print(f"  Max: {batch['bbox'].max().item()}")
print(f"  Any NaN: {torch.isnan(batch['bbox'].float()).any()}")
print(f"  Any > 1000: {(batch['bbox'] > 1000).any()}")

In [ ]:
print("📊 Pixel values:")
print(f"  Min: {batch['pixel_values'].min().item():.4f}")
print(f"  Max: {batch['pixel_values'].max().item():.4f}")
print(f"  Any NaN: {torch.isnan(batch['pixel_values']).any()}")

In [ ]:
training_args = TrainingArguments(
    output_dir                  = f'{MODELS_DIR}/docfusion-v3',
    num_train_epochs            = 5,
    per_device_train_batch_size = 4,
    per_device_eval_batch_size  = 4,
    learning_rate               = 5e-5,
    weight_decay                = 0.01,
    eval_strategy               = "epoch",
    save_strategy               = "epoch",
    load_best_model_at_end      = True,
    metric_for_best_model       = "f1",
    save_total_limit            = 2,
    logging_steps               = 50,
    fp16                        = False,
    push_to_hub                 = False,
    report_to                   = "none",
    optim                       = "adamw_torch"
)

trainer = Trainer(
    model           = model,
    args            = training_args,
    train_dataset   = train_dataset,
    eval_dataset    = val_dataset,
    compute_metrics = compute_metrics
)



In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

In [ ]:

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model  = model.to(device)


model.train()
batch  = next(iter(DataLoader(train_dataset, batch_size=2)))
inputs = {k: v.to(device) for k, v in batch.items()}

with torch.no_grad():
    outputs = model(**inputs)

print(f"Forward pass result:")
print(f"   Loss:   {outputs.loss.item():.4f}")
print(f"   Logits: {outputs.logits.shape}")

In [ ]:
import os
os.environ['XLA_USE_BF16'] = '1'

try:
    import torch_xla.core.xla_model as xm
    print(f"✅ TPU available: {xm.xla_device()}")
except:
    print("⚠️ torch_xla not available")

In [ ]:
trainer.train()

In [ ]:
train_dataset = DocFusionDataset(all_train_data)
val_dataset   = DocFusionDataset(all_val_data)

print(f"Train: {len(train_dataset)}")
print(f"Val:   {len(val_dataset)}")

In [ ]:
device = torch.device("cpu")
model  = model.to(device)

model.train()
batch  = next(iter(DataLoader(train_dataset, batch_size=2)))
inputs = {k: v.to(device) for k, v in batch.items()}

with torch.no_grad():
    outputs = model(**inputs)

print(f"Forward pass result:")
print(f"   Loss:   {outputs.loss.item():.4f}")
print(f"   Logits: {outputs.logits.shape}")

In [ ]:
batch = next(iter(DataLoader(train_dataset, batch_size=2)))
print("Batch keys:", list(batch.keys()))
for k, v in batch.items():
    print(f"  {k}: {v.shape}")

In [ ]:
# Test on one sample manually
sample = all_train_data[0]
print(f"Words:  {sample['words'][:3]}")
print(f"Boxes:  {sample['boxes'][:3]}")
print(f"Labels: {sample['labels'][:3]}")

# Try encoding
encoding = tokenizer(
    sample['words'],
    boxes=sample['boxes'],
    word_labels=sample['labels'],
    padding='max_length',
    truncation=True,
    max_length=512,
    return_tensors='pt',
)
print(f"\nEncoding keys: {list(encoding.keys())}")

In [ ]:
from transformers import LayoutLMTokenizer

tokenizer = LayoutLMTokenizer.from_pretrained(
    "microsoft/layoutlm-base-uncased"
)



sample = all_train_data[0]
encoding = tokenizer(
    sample['words'],
    boxes=sample['boxes'],
    word_labels=sample['labels'],
    padding='max_length',
    truncation=True,
    max_length=512,
    return_tensors='pt',
)
print(f"Encoding keys: {list(encoding.keys())}")
for k, v in encoding.items():
    print(f"  {k}: {v.shape}")


In [ ]:

sample = all_train_data[0]

print(f"Number of words:  {len(sample['words'])}")
print(f"Number of boxes:  {len(sample['boxes'])}")
print(f"Number of labels: {len(sample['labels'])}")
print(f"\nFirst word type: {type(sample['words'][0])}")
print(f"First box type:  {type(sample['boxes'][0])}")
print(f"First box:       {sample['boxes'][0]}")

In [ ]:
print(f"Tokenizer type: {type(tokenizer)}")
print(f"Tokenizer class: {tokenizer.__class__.__name__}")


import inspect
sig = inspect.signature(tokenizer.__call__)
print(f"\nTokenizer parameters: {list(sig.parameters.keys())}")

In [ ]:
from transformers import LayoutLMv2Processor, LayoutLMv2ForTokenClassification


processor = LayoutLMv2Processor.from_pretrained(
    "microsoft/layoutlmv2-base-uncased",
    apply_ocr=False
)

model = LayoutLMv2ForTokenClassification.from_pretrained(
    "microsoft/layoutlmv2-base-uncased",
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id
)



sample   = all_train_data[0]
encoding = processor(
    images=Image.new('RGB', (224, 224)),
    text=sample['words'],
    boxes=sample['boxes'],
    word_labels=sample['labels'],
    padding='max_length',
    truncation=True,
    max_length=512,
    return_tensors='pt',
)
print(f"Encoding keys: {list(encoding.keys())}")

In [ ]:
sample = all_train_data[0]
print(f"Keys: {list(sample.keys())}")
print(f"image_path: {sample.get('image_path')}")
print(f"image type: {type(sample.get('image'))}")

In [ ]:

has_image = sum(1 for s in all_train_data if s.get('image') is not None)
has_path  = sum(1 for s in all_train_data if s.get('image_path') is not None)

print(f"Samples with PIL image: {has_image}")
print(f"Samples with image path: {has_path}")

In [ ]:
from transformers import BertTokenizerFast
import torch

tokenizer = BertTokenizerFast.from_pretrained(
    "microsoft/layoutlm-base-uncased"
)

model = LayoutLMForTokenClassification.from_pretrained(
    "microsoft/layoutlm-base-uncased",
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id
)


In [ ]:
def process_sample(sample):
    words  = sample['words']
    boxes  = sample['boxes']
    labels_list = sample['labels']


    tokenized = tokenizer(
        words,
        is_split_into_words=True,
        padding='max_length',
        truncation=True,
        max_length=512,
        return_tensors='pt'
    )


    word_ids    = tokenized.word_ids(batch_index=0)
    token_boxes = []
    token_labels = []

    for word_idx in word_ids:
        if word_idx is None:
            token_boxes.append([0, 0, 0, 0])
            token_labels.append(-100)
        else:
            token_boxes.append(boxes[word_idx])
            token_labels.append(labels_list[word_idx])


    tokenized['bbox']   = torch.tensor([token_boxes],  dtype=torch.long)
    tokenized['labels'] = torch.tensor([token_labels], dtype=torch.long)

    return {k: v.squeeze(0) for k, v in tokenized.items()}

In [ ]:
sample   = all_train_data[0]
encoding = process_sample(sample)

print("Keys:", list(encoding.keys()))
for k, v in encoding.items():
    print(f"  {k}: {v.shape}")


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model  = model.to(device)
model.train()

train_dataset = DocFusionDataset(all_train_data)
val_dataset   = DocFusionDataset(all_val_data)

batch  = next(iter(DataLoader(train_dataset, batch_size=2)))
inputs = {k: v.to(device) for k, v in batch.items()}

with torch.no_grad():
    outputs = model(**inputs)

print(f"Forward pass result:")
print(f"   Loss:   {outputs.loss.item():.4f}")
print(f"   Logits: {outputs.logits.shape}")

In [ ]:

model = model.to(torch.bfloat16)
model = model.to(device)
print(f"Model dtype: {next(model.parameters()).dtype}")

In [ ]:
training_args = TrainingArguments(
    output_dir                  = f'{MODELS_DIR}/docfusion-v1',
    num_train_epochs            = 5,
    per_device_train_batch_size = 8,
    per_device_eval_batch_size  = 8,
    learning_rate               = 2e-5,
    weight_decay                = 0.01,
    eval_strategy               = "epoch",
    save_strategy               = "epoch",
    load_best_model_at_end      = True,
    metric_for_best_model       = "f1",
    save_total_limit            = 2,
    logging_steps               = 50,
    fp16                        = False,
    bf16                        = True,
    push_to_hub                 = False,
    report_to                   = "none",
    optim                       = "adamw_torch",
    warmup_ratio                = 0.1,
    max_grad_norm               = 1.0,
    dataloader_num_workers      = 0,
)

trainer = Trainer(
    model           = model,
    args            = training_args,
    train_dataset   = train_dataset,
    eval_dataset    = val_dataset,
    compute_metrics = compute_metrics
)



In [ ]:
sample = train_dataset[0]
unique_labels = sample['labels'].unique().tolist()
print(f"Unique labels in sample: {unique_labels}")


In [ ]:
import os
os.environ['PJRT_DEVICE'] = 'TPU'

import torch_xla.core.xla_model as xm
device = xm.xla_device()
print(f"✅ TPU device: {device}")

In [ ]:
trainer.train()

In [ ]:
model = LayoutLMForTokenClassification.from_pretrained(
    "microsoft/layoutlm-base-uncased",
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id,
    torch_dtype=torch.bfloat16
)
model = model.to(device)
print(f"Model dtype: {next(model.parameters()).dtype}")

In [ ]:
model.train()
batch  = next(iter(DataLoader(train_dataset, batch_size=2)))
inputs = {k: v.to(device) for k, v in batch.items()}

with torch.no_grad():
    outputs = model(**inputs)

xm.mark_step()
print(f"Loss: {outputs.loss.item():.4f}")

In [ ]:
training_args = TrainingArguments(
    output_dir                  = f'{MODELS_DIR}/docfusion-v2',
    num_train_epochs            = 5,
    per_device_train_batch_size = 8,
    per_device_eval_batch_size  = 8,
    learning_rate               = 2e-5,
    weight_decay                = 0.01,
    eval_strategy               = "epoch",
    save_strategy               = "epoch",
    load_best_model_at_end      = True,
    metric_for_best_model       = "f1",
    save_total_limit            = 2,
    logging_steps               = 50,
    fp16                        = False,
    bf16                        = True,
    push_to_hub                 = False,
    report_to                   = "none",
    optim                       = "adamw_torch",
    warmup_ratio                = 0.1,
    max_grad_norm               = 1.0,
    dataloader_num_workers      = 0,
)

trainer = Trainer(
    model           = model,
    args            = training_args,
    train_dataset   = train_dataset,
    eval_dataset    = val_dataset,
    compute_metrics = compute_metrics
)



In [ ]:
trainer.train()

In [ ]:


model.push_to_hub("Zakariya007/docfusion-v1")
tokenizer.push_to_hub("Zakariya007/docfusion-v1")
print("Pushed to HuggingFace Hub!")

In [ ]:
import copy


model_cpu = model.to('cpu')
print("Model moved to CPU!")


model_cpu.save_pretrained(f'{MODELS_DIR}/docfusion-v1')
tokenizer.save_pretrained(f'{MODELS_DIR}/docfusion-v1')
print("Saved to Drive!")


model_cpu.push_to_hub("Zakariya007/docfusion-v1")
tokenizer.push_to_hub("Zakariya007/docfusion-v1")
print("Pushed to HuggingFace Hub!")

In [ ]:

model.save_pretrained(f'{MODELS_DIR}/docfusion-v1')
tokenizer.save_pretrained(f'{MODELS_DIR}/docfusion-v1')


In [ ]:

import os

PROJECT_DIR = '/content/drive/MyDrive/DocFusion'
MODELS_DIR  = f'{PROJECT_DIR}/models'


checkpoints = os.listdir(f'{MODELS_DIR}/docfusion-v2')
print(f"Checkpoints: {checkpoints}")

In [ ]:
PROJECT_DIR = '/content/drive/MyDrive/DocFusion'
MODELS_DIR  = f'{PROJECT_DIR}/models'

In [ ]:
from transformers import LayoutLMForTokenClassification, BertTokenizerFast

checkpoint_path = f'{MODELS_DIR}/docfusion-v2/checkpoint-895'

model     = LayoutLMForTokenClassification.from_pretrained(checkpoint_path)
tokenizer = BertTokenizerFast.from_pretrained("microsoft/layoutlm-base-uncased")

print(f"Model loaded from checkpoint-895!")
print(f"   Num labels: {model.config.num_labels}")

In [ ]:
model = model.to('cpu')

model.push_to_hub("Zakariya007/docfusion-v1")
tokenizer.push_to_hub("Zakariya007/docfusion-v1")
print("Pushed to HuggingFace Hub!")